# Mean Reversion Strategy, Backtested on Crypto
Mean reversion bets that a stretched price snaps back to its average. The rule most tutorials teach:
the z-score $z = \dfrac{\text{close} - \text{mean}}{\text{standard deviation}}$ over the last L candles; buy when
$z < -K$, sell short when $z > +K$, exit when $z$ is back at 0. The idea has academic roots — short-term
reversal on stocks: Lehmann (1990, Quarterly Journal of Economics), Jegadeesh (1990, Journal of
Finance). We test it on eleven perpetual futures, hourly candles, with fees and funding, from 2021 to
October 2026 — and then the honest way: tune it on 2021–2023, test it on 2024–2026, which it has never
seen. Education, not financial advice.

## 1. Install
freqtrade is the backtester; we call only a few of its methods.

In [ ]:
!pip install -q freqtrade==2026.10

## 2. Helpers from the earlier episodes
`binance_vision` — raw Binance data with SHA-256 hash checks (episode q00); `quant_report` — the verdict and hourly marked-to-market equity (episode q01).

In [ ]:
!curl -fsSLO https://raw.githubusercontent.com/nikshev/unstuck/main/episodes/quant-lib/binance_vision.py -O https://raw.githubusercontent.com/nikshev/unstuck/main/episodes/quant-lib/quant_report.py
from binance_vision import save_for_freqtrade, DATA
from quant_report import verdict, equity, drawdown, newest
from freqtrade.data.btanalysis import load_backtest_stats
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 3. The coins, every hour
The same eleven as episodes q02 and q03: every perpetual Binance listed by January 2020. Hourly candles, mark prices and funding from 2020. Uses `save_for_freqtrade`, `DATA` (section 2).

In [ ]:
coins = ["BTC", "ETH", "BCH", "XRP", "EOS", "LTC",
         "TRX", "ETC", "LINK", "XLM", "ADA"]
for c in coins:
    save_for_freqtrade(c + "USDT", "1h", "2020-01", "2026-10")
hourly = str(DATA / "{}_USDT_USDT-1h-futures.feather")
close = pd.DataFrame({c: pd.read_feather(hourly.format(c))
                         .set_index("date")["close"] for c in coins})
print(close.notna().sum().to_string())                 # hours of data per coin

## 4. Does a stretched price snap back?
The z-score over L = 20 hours: how many standard deviations the close is from its 20-hour mean. If mean reversion is real, the price should fall after $z > +2$ and rise after $z < -2$. Bitcoin, 2021 to October 2026, the change over the next 20 hours. Uses `close` (section 3).

In [ ]:
btc = close["BTC"].loc["2021":]
z = (btc - btc.rolling(20).mean()) / btc.rolling(20).std()
nxt = btc.shift(-20) / btc - 1                  # change over the next 20 h
for name, m in (("z below -2", z < -2), ("z above +2", z > 2),
                ("every hour", z.notna())):
    print(f"{name:>11}: {m.mean():5.1%} of hours,"
          f" next 20 h {nxt[m].mean():+.3%}, up {(nxt[m] > 0).mean():.0%}")

## 5. Backtest settings
Isolated margin, leverage 1, hourly candles, ten slots of equal size — one per coin freqtrade will trade. 10,000 USDT (Tether, a dollar stablecoin), a 0.05% fee per order. Uses `coins` (section 3).

In [ ]:
config = {
    "trading_mode": "futures",
    "margin_mode": "isolated",               # each position has its own margin
    "liquidation_buffer": 0.0,
    "stake_currency": "USDT",
    "stake_amount": "unlimited",             # account / free slots
    "dry_run_wallet": 10000, "dry_run": True,
    "max_open_trades": 10,                   # one slot per coin traded
    "timeframe": "1h", "fee": 0.0005,
    "entry_pricing": {"price_side": "same", "use_order_book": True,
                      "order_book_top": 1},
    "exit_pricing": {"price_side": "same", "use_order_book": True,
                     "order_book_top": 1},
    "exchange": {"name": "binance",
                 "pair_whitelist": [c + "/USDT:USDT" for c in coins]},
    "pairlists": [{"method": "StaticPairList"}],
}
json.dump(config, open("config.json", "w"), indent=1)

## 6. Strategy file, part 1: the settings
The textbook values: L = 20 candles, K = 2 standard deviations, both sides. No stop loss — the exit is the mean — so the stop is freqtrade's minimum, −99%.

In [ ]:
%%writefile user_data/strategies/MeanReversion.py
from freqtrade.strategy import IStrategy

class MeanReversion(IStrategy):
    L, K, SHORTS = 20, 2.0, True        # lookback, threshold, shorts on
    timeframe = "1h"
    can_short = True
    minimal_roi = {"0": 100}            # no profit target: only the exit rule
    stoploss = -0.99                    # the textbook rule has no stop
    startup_candle_count = 100          # the longest lookback we try

## 7. Part 2: the z-score
The same three lines as section 4, with `L` from section 6.

In [ ]:
%%writefile -a user_data/strategies/MeanReversion.py

    def populate_indicators(self, df, metadata):
        mean = df["close"].rolling(self.L).mean()
        std = df["close"].rolling(self.L).std()
        df["z"] = (df["close"] - mean) / std        # distance in std units
        return df

## 8. Part 3: entries and exits
Signals on the hourly close; the order fills at the next candle's open. `K`, `SHORTS` from section 6, `z` from 7.

In [ ]:
%%writefile -a user_data/strategies/MeanReversion.py

    def populate_entry_trend(self, df, metadata):
        df.loc[df["z"] < -self.K, "enter_long"] = 1     # stretched down
        if self.SHORTS:
            df.loc[df["z"] > self.K, "enter_short"] = 1  # stretched up
        return df

    def populate_exit_trend(self, df, metadata):
        df.loc[df["z"] >= 0, "exit_long"] = 1           # back at the mean
        df.loc[df["z"] <= 0, "exit_short"] = 1
        return df

## 9. Run it
1 January 2021 to 31 October 2026. Uses `config.json` (5), `MeanReversion` (6–8), `verdict`, `equity`, `drawdown` (2).

In [ ]:
!freqtrade backtesting --cache none -c config.json -s MeanReversion --timerange 20210101-20261031 --export trades > run.log 2>&1
both, trades = verdict("MeanReversion")
both["drawdown, marked to market"] = drawdown(equity(trades,
                                                     start="2021-01-01"))

## 10. What a trade looks like
Win rate, average win and loss, hours held — by side; then the count and the money of the wins and the losses. Uses `trades` (section 9).

In [ ]:
trades["side"] = np.where(trades["is_short"], "short", "long")
trades["hours"] = trades["trade_duration"] / 60          # minutes -> hours
trades["win"] = trades["profit_abs"] > 0
stats = trades.groupby("side").agg(
    trades=("win", "size"), win_rate=("win", "mean"),
    avg_win=("profit_abs", lambda p: p[p > 0].mean()),
    avg_loss=("profit_abs", lambda p: p[p <= 0].mean()),
    hours=("hours", "mean"))
print(stats.round(2).to_string())
won_lost = trades.groupby(["side", "win"])["profit_abs"].agg(["size", "sum"])
print(won_lost.round(0).to_string())          # how many, and how much money

### Where the money was lost: by coin
Profit of every trade added up per coin and side. Uses `trades` (section 9), `side` (10).

In [ ]:
by_coin = trades.pivot_table("profit_abs", "pair", "side", aggfunc="sum")
by_coin.index = by_coin.index.str.split("/").str[0]     # BTC/USDT:USDT -> BTC
by_coin["total"] = by_coin["long"] + by_coin["short"]
by_coin = by_coin.rename_axis(index=None, columns=None)
print(by_coin.sort_values("total").round().astype(int).to_string())

### And by year
The same profit, by the year the trade closed. Uses `trades` (9), `side` (10).

In [ ]:
year = trades["close_date"].dt.year.rename("year")
by_year = trades.pivot_table("profit_abs", year, "side", aggfunc="sum")
by_year["total"] = by_year["long"] + by_year["short"]
by_year = by_year.rename_axis(index=None, columns=None)
print(by_year.round().astype(int).to_string())

## 11. Tune it: nine settings, long only, 2021–2023
Shorts lost (section 10), so long only. L = 20, 50, 100 hours × K = 1.5, 2, 2.5, on the in-sample years only. freqtrade loads the strategy file fresh for every run, so the two numbers travel in the environment. Uses `MeanReversion` (6–8), `newest` (2).

In [ ]:
%%writefile user_data/strategies/LongOnly.py
import os
from MeanReversion import MeanReversion

class LongOnly(MeanReversion):
    SHORTS = False
    L = int(os.environ.get("L", 20))          # set by the loop below
    K = float(os.environ.get("K", 2.0))

### Run the nine
CAGR (compound annual growth rate, % a year) and the number of trades of each, read from freqtrade's summary. Uses `LongOnly` (above), `config.json` (5), `newest`, `load_backtest_stats` (2).

In [ ]:
def run_grid(timerange):
    cagr, count = {}, {}
    for L in (20, 50, 100):
        for K in (1.5, 2.0, 2.5):
            os.environ["L"], os.environ["K"] = str(L), str(K)
            !freqtrade backtesting --cache none -c config.json -s LongOnly --timerange {timerange} --export trades > run.log 2>&1
            s = load_backtest_stats(newest())["strategy"]["LongOnly"]
            cagr[(L, K)] = 100 * s["cagr"]            # % a year
            count[(L, K)] = s["total_trades"]
    return pd.Series(cagr).unstack(), pd.Series(count).unstack()

in_sample, in_trades = run_grid("20210101-20231231")
best = in_sample.stack().idxmax()             # picked by the code, not by me
print("% a year:"); print(in_sample.round(1).to_string())
print("trades:"); print(in_trades.to_string())
print("best in-sample: L =", best[0], " K =", best[1])

## 12. The test it has never seen: 2024 to October 2026
The same nine settings on the out-of-sample years. Uses `run_grid`, `in_sample`, `best` (section 11).

In [ ]:
out_sample, out_trades = run_grid("20240101-20261031")
print("% a year:"); print(out_sample.round(1).to_string())
rank = out_sample.stack().rank(ascending=False)
print(f"best in-sample: {in_sample.loc[best]:+.1f}% a year in 2021-23,"
      f" {out_sample.loc[best]:+.1f}% in 2024-26")
print(f"its rank out of sample: {rank[best]:.0f} of 9")
top = out_sample.stack().idxmax()             # the best on 2024-26
print(f"best out of sample: L = {top[0]}, K = {top[1]}:"
      f" {out_sample.loc[top]:+.1f}%, in sample {in_sample.loc[top]:+.1f}%")

## 13. The picked setting, January 2021 to October 2026
The in-sample winner over the whole period: the tuned years and the unseen ones together. Uses `best` (11), `LongOnly` (11), `config.json` (5), `verdict` (2).

In [ ]:
os.environ["L"], os.environ["K"] = str(best[0]), str(best[1])
!freqtrade backtesting --cache none -c config.json -s LongOnly --timerange 20210101-20261031 --export trades > run.log 2>&1
picked, picked_trades = verdict("LongOnly")

### Against holding the same coins
Buy-and-hold = the ten coins freqtrade traded, equal amounts, rebalanced daily; the dashed line = 1 January 2024, where the unseen years begin. Uses `close` (3), `trades` (9), `picked`, `picked_trades` (above), `equity`, `drawdown` (2).

In [ ]:
traded = close.drop(columns="EOS").resample("1D").last().loc["2021-01-01":]
hold = 10_000 * (1 + traded.pct_change().mean(axis=1).fillna(0)).cumprod()
curves = {"buy and hold, 10 coins": hold,
          "z-score, both sides": equity(trades, start="2021-01-01"),
          "long only, picked": equity(picked_trades, start="2021-01-01")}
picked["drawdown, marked to market"] = drawdown(curves["long only, picked"])
for name, curve in curves.items():
    print(f"{name:>22}: final {curve.iloc[-1]:>7,.0f}"
          f"  max drawdown {drawdown(curve):.0%}")
plt.figure(figsize=(11, 5))
for name, curve in curves.items():
    plt.plot(curve.index, curve, label=name)
plt.axvline(pd.Timestamp("2024-01-01", tz="UTC"), color="grey", ls="--")
plt.yscale("log"); plt.legend(); plt.title("10,000 USDT, 2021 to 2026")
plt.show()

### Year by year
Each account's value at the end of every year, and its change that year; 2021 starts from 10,000 USDT (Tether, a dollar stablecoin). Uses `curves` (above).

In [ ]:
ends = pd.DataFrame({n: c.resample("YE").last() for n, c in curves.items()})
ends.columns = ["hold", "both sides", "picked"]
ends.index = list(ends.index.year)
change = 100 * (ends / ends.shift().fillna(10_000) - 1)    # % that year
print("value at the end of the year:")
print(ends.round().astype(int).to_string())
print("change that year, %:")
print(change.round(1).to_string())

## 14. The Sharpe ratio: return per unit of wobble
The same four lines as episode q03: Sharpe ratio $S = \dfrac{\text{mean of the daily returns}}{\text{their standard deviation}} \times \sqrt{365}$; risk-free rate 0: idle USDT (Tether, a dollar stablecoin) earns nothing here. The picked setting also split at 1 January 2024. Uses `curves` (13), `both` (9), `picked` (13).

In [ ]:
def sharpe(curve):
    day = curve.resample("1D").last()           # the account once a day
    r = day.pct_change().dropna()               # daily returns
    return r.mean() / r.std() * 365 ** 0.5      # a year = 365 crypto days

for name, curve in curves.items():
    r = curve.resample("1D").last().pct_change().dropna()
    print(f"{name:>22}: mean {r.mean():.3%}  std {r.std():.2%}"
          f"  Sharpe {sharpe(curve):.2f}")
p = curves["long only, picked"]
print(f"picked, 2021-23: {sharpe(p.loc[:'2023']):.2f};"
      f" 2024-26: {sharpe(p.loc['2024':]):.2f}")
both["Sharpe"] = sharpe(curves["z-score, both sides"])
picked["Sharpe"] = sharpe(p)

## 15. Every run side by side
A legend explains every row. Uses `both` (9, 14), `picked` (13, 14).

In [ ]:
table = pd.DataFrame({"both sides": both, "long, picked": picked})
table.index = ["start", "final", "CAGR", "drawdown (freqtrade)", "trades",
               "long profit", "short profit", "funding", "fees",
               "drawdown (hourly)", "Sharpe"]
pct = ["CAGR", "drawdown (freqtrade)", "drawdown (hourly)"]
def cell(row, v):                       # how each row is written
    if row in pct:
        return f"{v:.0%}"
    return f"{v:.2f}" if row == "Sharpe" else f"{v:,.0f}"
shown_table = table.apply(lambda r: r.map(lambda v: cell(r.name, v)), axis=1)
print(shown_table.to_string())
print("money in USDT (Tether, a dollar stablecoin), 2021 - Oct 2026")
print("both sides: L = 20, K = 2; picked: long only, best in 2021-23")
print("CAGR = compound annual growth rate: growth per year")
print("drawdown = deepest fall from a peak; freqtrade: closed trades only,")
print("  hourly: the whole account valued every hour")
print("funding = paid between longs and shorts every 8 hours")
print("fees = the exchange fee, 0.05% of every order")
print("Sharpe = mean daily return / its standard deviation x sqrt(365);")
print("  above 1 is the usual target")